# IV / 그룹기여도(leave-one-out) — 최종 A·B·C 기준

전처리 완료본 3개를 병합해 변수별 **IV(Information Value)** 와 개념그룹별 **조건부 기여도**를 산출한다.

| 입력 | 경로 |
| --- | --- |
| A 금융 | `데이터/완료/재계산/model_A_financial_final.csv` |
| B 비금융(기존) | `데이터/완료/재계산/model_B_nonfinancial_final.csv` |
| C 비금융(신규) | `데이터/완료/재계산/model_C_nonfinancial_new_final.csv` |

`SK_ID_CURR` + `TARGET` 기준 inner join. `AGE_BAND`는 EDA 전용이라 모델 입력에서 제외한다.

---

## ⚠️ 재현 범위 — 반드시 읽을 것

이 노트북은 **변수선정이 끝난 최종 64개(A18 / B31 / C15) 기준**이다.
원본 버전은 선정 *이전* 83피처(`model_dataset_v2.csv`)로 돌아갔고, 입력을 바꾸면서 아래가 달라진다.

**재현되는 것** — 최종 64개 변수의 IV, 개념그룹별 기여도, 5C 매핑

**재현되지 않는 것** — 최종본에서 이미 제외된 변수들의 판정 근거.
구체적으로 `FLAG_OWN_REALTY_BIN`(그룹기여도 −0.00007), `CNT_CHILDREN`(기여도 음수),
`FLAG_CONT_MOBILE`(99.81% 동일값), `DTI`, `AMT_INCOME_OUTLIER`, `AMT_REQ_CREDIT_BUREAU_*` 는
A·B·C 파일에 아예 없으므로 이 노트북으로는 "왜 뺐는지"를 보여줄 수 없다.

→ PPT 슬라이드 8의 제외 사유(자가보유·자녀수)를 코드로 뒷받침하려면
   `model_dataset_v2.csv` 기준 원본 실행 결과를 함께 보관해야 한다.

## EXT_SOURCE 취급

EXT는 **모델 입력이 아니다.** A·B·C 파일에도 들어 있지 않다.
IV 표의 `EXT2_상관` 컬럼은 "이 변수가 외부신용점수와 얼마나 겹치나 = 얼마나 새로운 정보인가"를
재는 **진단 지표**일 뿐이며, PPT 슬라이드 8의 "EXT를 뺀 이유"를 뒷받침하는 근거다.

EXT 파일을 못 찾으면 해당 컬럼만 비우고 나머지는 정상 실행된다.


## 0. 구글드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1. 단변량 분석 — 설정 (경로/상수)

In [ ]:
"""
step1_univariate — 사전 변수분석 (단변량) [최종 A·B·C 기준]

목적: M1~M4 모델링 전에, 각 변수가 TARGET과 얼마나 관련 있는지(IV),
      그 정보가 외부신용점수(EXT_SOURCE)와 겹치는지(상관)를 확인한다.

주 지표는 IV(Information Value).

입력 : model_A_financial_final.csv / model_B_nonfinancial_final.csv
       / model_C_nonfinancial_new_final.csv   (데이터/완료/재계산)
출력 : outputs_uni/univariate_summary.csv     (변수별 IV·검정·EXT상관 통합표)
"""

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")

# 본인 환경에 맞게 수정
BASE_DIR = Path("/content/drive/MyDrive/BOOSTMAP/데이터/완료/재계산")

FILES = {
    "A_금융":        BASE_DIR / "model_A_financial_final.csv",
    "B_비금융_기존":  BASE_DIR / "model_B_nonfinancial_final.csv",
    "C_비금융_신규":  BASE_DIR / "model_C_nonfinancial_new_final.csv",
}

OUT = BASE_DIR / "outputs_uni"; OUT.mkdir(parents=True, exist_ok=True)

TARGET = "TARGET"
ID = "SK_ID_CURR"

# 모델 입력에서 빼는 컬럼 — AGE_BAND는 AGE와 중복이며 EDA·시각화 전용
DROP_COLS = {"AGE_BAND"}

# EXT_SOURCE_2 조달 경로 (없으면 EXT2_상관만 공란 처리하고 계속 진행)
EXT_COL = "EXT_SOURCE_2"
EXT_CANDIDATES = [
    BASE_DIR / "model_EXT_source.csv",
    BASE_DIR.parent / "model_EXT_source.csv",
    Path("/content/drive/MyDrive/BOOSTMAP/데이터/원본/application_train.csv"),
]

N_BINS = 10        # 연속변수 구간 수
EPS = 0.5          # WOE 라플라스 보정(0 셀 방지)

### A·B·C 병합

변수의 소속 군(A/B/C)을 하드코딩된 목록이 아니라 **어느 파일에서 왔는지**로 판정한다.
전처리 결과가 바뀌어도 분류가 따라오므로 원본 방식(`A_FEATURES` 집합 하드코딩)보다 안전하다.

In [ ]:
def load_abc(verbose=True):
    """A·B·C 완료본을 SK_ID_CURR+TARGET 기준으로 병합. (df, 변수→소속군 dict) 반환."""
    df, origin = None, {}

    for grp, path in FILES.items():
        if not path.exists():
            raise FileNotFoundError(f"{grp} 파일을 찾을 수 없음: {path}")
        d = pd.read_csv(path)

        missing = {ID, TARGET} - set(d.columns)
        if missing:
            raise KeyError(f"{grp}: 필수 컬럼 없음 {missing}")

        feats = [c for c in d.columns if c not in (ID, TARGET) and c not in DROP_COLS]
        for c in feats:
            if c in origin:
                raise KeyError(f"컬럼 중복: '{c}' ({origin[c]} / {grp})")
            origin[c] = grp

        d = d[[ID, TARGET] + feats]
        df = d if df is None else df.merge(d, on=[ID, TARGET], how="inner")

        if verbose:
            print(f"  {grp:12s} {len(feats):3d}개 변수  ({path.name})")

    if verbose:
        print(f"\n병합 결과: {df.shape[0]:,}행 / 피처 {df.shape[1] - 2}개 "
              f"/ 부도율 {df[TARGET].mean():.2%}")
        # 검산값 대조 — 슬라이드 10 기준
        if df.shape[0] != 307_511 or int(df[TARGET].sum()) != 24_825:
            print(f"  ⚠️ 검산 불일치: 기대 307,511행 / 연체 24,825 "
                  f"→ 실제 {df.shape[0]:,}행 / {int(df[TARGET].sum()):,}")

    return df, origin


def load_ext(ids):
    """EXT_SOURCE_2를 붙인다. 못 찾으면 None (해당 지표만 생략)."""
    for p in EXT_CANDIDATES:
        if not p.exists():
            continue
        try:
            e = pd.read_csv(p, usecols=[ID, EXT_COL])
        except (ValueError, KeyError):
            continue
        print(f"EXT 참조: {p}")
        return e.set_index(ID)[EXT_COL].reindex(ids).values
    print("EXT 파일 없음 → EXT2_상관은 공란으로 둡니다. (분석 자체는 정상 진행)")
    return None

### 변수 분류 (군 · 자료형)

In [ ]:
def classify_columns(df, origin):
    rows = []
    for c in df.columns:
        if c in (ID, TARGET):
            continue
        u = df[c].dropna().unique()
        is_bin = set(np.unique(u)).issubset({0, 1}) and len(u) <= 2
        rows.append({"feature": c,
                     "group": origin.get(c, "미분류"),
                     "vtype": "binary" if is_bin else "continuous"})
    return pd.DataFrame(rows)

### WOE / IV 계산 함수

In [ ]:
# --------------------------------------------------------- WOE / IV
def _woe_iv_from_groups(g, y):
    df = pd.DataFrame({"g": g, "y": y})
    tot_bad = df.y.sum(); tot_good = len(df) - tot_bad
    agg = df.groupby("g").agg(n=("y", "size"), bad=("y", "sum"))
    agg["good"] = agg.n - agg.bad
    agg["bad_dist"] = (agg.bad + EPS) / (tot_bad + EPS * len(agg))
    agg["good_dist"] = (agg.good + EPS) / (tot_good + EPS * len(agg))
    agg["woe"] = np.log(agg.good_dist / agg.bad_dist)
    agg["iv_part"] = (agg.good_dist - agg.bad_dist) * agg.woe
    agg["bad_rate"] = agg.bad / agg.n
    return float(agg.iv_part.sum()), agg.reset_index()


def woe_iv_binary(x, y):
    return _woe_iv_from_groups(x.fillna(-1), y)


def woe_iv_continuous(x, y, n_bins=N_BINS):
    # 결측은 반드시 별도 구간으로 유지 (결측 자체가 신호)
    x = x.replace([np.inf, -np.inf], np.nan)
    binned = pd.Series(index=x.index, dtype=object)
    notna = x.notna()
    try:
        binned[notna] = pd.qcut(x[notna], n_bins, duplicates="drop").astype(str)
    except ValueError:
        binned[notna] = pd.cut(x[notna], min(n_bins, x[notna].nunique())).astype(str)
    binned[~notna] = "MISSING"
    return _woe_iv_from_groups(binned, y)


def iv_strength(iv):
    if iv < 0.02: return "무용"
    if iv < 0.1:  return "약함"
    if iv < 0.3:  return "중간"
    if iv < 0.5:  return "강함"
    return "매우강함(과적합 의심)"

### 유의성 검정 (카이제곱 / Mann-Whitney)

In [ ]:
# --------------------------------------------------------- 유의성 검정
def chi2_binary(x, y):
    ct = pd.crosstab(x.fillna(-1), y)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))     # Cramér's V
    return chi2, p, v


def mwu_continuous(x, y):
    # 소득/대출액은 극단적 우편향 → 정규성 가정 없는 Mann-Whitney가 안전
    x = x.replace([np.inf, -np.inf], np.nan)
    a = x[(y == 0) & x.notna()]; b = x[(y == 1) & x.notna()]
    if len(a) < 2 or len(b) < 2:
        return np.nan, np.nan, np.nan
    u, p_mwu = stats.mannwhitneyu(a, b, alternative="two-sided")
    rbc = 1 - 2 * u / (len(a) * len(b))               # rank-biserial 효과크기
    t, p_t = stats.ttest_ind(a, b, equal_var=False)   # 참고용 Welch t
    return p_mwu, rbc, p_t

### EXT_SOURCE 상관 (신규성 진단)

EXT는 모델 입력이 아니다. 이 지표는 **PPT 슬라이드 8 "EXT 외부점수를 뺀 이유"** 의 근거로만 쓴다.
EXT_SOURCE_2를 쓰는 이유는 결측이 0.2%로 3개 중 가장 온전하기 때문이다. 낮을수록 새로운 정보다.

In [ ]:
# --------------------------------------------------------- EXT 상관(신규성)
def ext_correlation(x, ext, vtype):
    if ext is None:
        return np.nan
    ext = pd.Series(ext, index=x.index)
    m = ext.notna() & x.notna() & np.isfinite(x.replace([np.inf, -np.inf], np.nan))
    if m.sum() < 100:
        return np.nan
    if vtype == "continuous":
        r, _ = stats.spearmanr(x[m], ext[m])
    else:
        r, _ = stats.pointbiserialr(x[m].astype(int), ext[m])
    return abs(r)

### 실행 (main)

In [ ]:
# --------------------------------------------------------- 메인
def main():
    df, origin = load_abc()
    y = df[TARGET].values
    ext = load_ext(df[ID])

    cls = classify_columns(df, origin)
    print()
    print(cls.group.value_counts().to_string())

    summary, woe_tables = [], {}
    for _, row in cls.iterrows():
        f, grp, vt = row.feature, row.group, row.vtype
        x = df[f]
        if vt == "binary":
            iv, wtab = woe_iv_binary(x, y)
            _, p_assoc, effect = chi2_binary(x, y); test = "chi2"
        else:
            iv, wtab = woe_iv_continuous(x, y)
            p_assoc, effect, _ = mwu_continuous(x, y); test = "mann_whitney"
        woe_tables[f] = wtab
        ext_corr = ext_correlation(x, ext, vt)
        summary.append({"feature": f, "group": grp, "vtype": vt,
                        "IV": round(iv, 4), "IV_강도": iv_strength(iv),
                        "검정": test, "p_value": p_assoc,
                        "효과크기": round(effect, 4) if pd.notna(effect) else np.nan,
                        "EXT2_상관": round(ext_corr, 4) if pd.notna(ext_corr) else np.nan})

    S = pd.DataFrame(summary).sort_values("IV", ascending=False)
    S["유의_5pct"] = S.p_value < 0.05
    S.to_csv(OUT / "univariate_summary.csv", index=False, encoding="utf-8-sig")
    print(f"\n저장 완료: {OUT / 'univariate_summary.csv'} ({len(S)}개 변수)")
    print("\nIV 상위 15개")
    print(S.head(15)[["feature", "group", "IV", "IV_강도", "EXT2_상관"]].to_string(index=False))
    return S


if __name__ == "__main__":
    S = main()

## Step 2. 그룹기여도(leave-one-out) & 변수 판정 — 설정

In [ ]:
"""
step2_group_contribution — 그룹 단위 조건부 기여도 [최종 A·B·C 기준]

배경: IV는 변수를 '혼자' 평가하지만, 실제 모델은 변수를 조합해서 쓴다.
      그래서 leave-one-out(하나 빼고 AUC 재측정)으로 '다른 변수가 다 있을 때의
      추가 기여'를 측정한다. 이게 중복 변수를 걸러낸다.

핵심 원칙:
  1. 원핫 더미(직업/업종/소득유형/교육수준/주거유형)는 개별이 아니라 '그룹 통째로' 넣고 뺀다.
  2. 원본 금액(AMT_*)은 비율변수(ANNUITY_TO_CREDIT 등)와 정보가 겹친다 → 대표만 유지.
  3. 상관 높은 쌍(EDA에서 발견, |r|>=0.9)의 그룹기여도를 특히 주의 깊게 볼 것:
     - DAYS_EMPLOYED_ANOM ↔ NAME_INCOME_TYPE_G_Pensioner (r=0.9996)
       → B군 SQL에서 55,374명이 정확히 같은 사람이라고 처리한 근거가 여기서 나왔다.

입력 : A·B·C 완료본, outputs_uni/univariate_summary.csv
출력 : outputs_uni/group_decision.csv   (개념그룹별 판정)
       outputs_uni/변수선정_최종.csv     (변수 단위 포함/제외 + 근거)
"""

import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

warnings.filterwarnings("ignore")

SAMPLE_N = 150000          # 속도용 표본 (전체 30만이면 시간 오래 걸림). None이면 전체 사용
RANDOM_STATE = 42

### 개념그룹 정의 (build_groups)

원본과 달리 **존재하는 컬럼만 그룹에 담는다.** 최종본에 없는 변수(`DTI`, `AMT_INCOME_OUTLIER`,
`FLAG_OWN_REALTY_BIN`, `AMT_REQ_CREDIT_BUREAU_*`, `CNT_CHILDREN`, `FLAG_CONT_MOBILE`)를
참조하던 자리가 비면 그룹 자체를 자동으로 뺀다. 원본은 여기서 `KeyError`가 났다.

In [ ]:
def build_groups(allcols):
    """변수를 '개념 그룹'으로 묶는다. 원핫은 접두사로 묶음. 없는 변수는 자동 제외."""
    cols = set(allcols)
    def pick(*names):  return [c for c in names if c in cols]
    def pref(*ps):     return [c for c in allcols if c.startswith(ps)]

    g = {
        # ---- A_금융 ----
        "ANNUITY_TO_CREDIT(상환부담)":   pick("ANNUITY_TO_CREDIT"),
        "CREDIT_TO_INCOME(대출/소득)":   pick("CREDIT_TO_INCOME"),
        "LTV_GOODS(담보비율)":           pick("LTV_GOODS"),
        "NAME_CONTRACT_TYPE(대출유형)":  pick("NAME_CONTRACT_TYPE_BIN"),
        "AMT원본_금액군(중복후보)":       pick("AMT_CREDIT", "AMT_CREDIT_LOG",
                                          "AMT_ANNUITY", "AMT_ANNUITY_LOG",
                                          "AMT_GOODS_PRICE", "AMT_GOODS_PRICE_LOG",
                                          "AMT_INCOME_TOTAL", "AMT_INCOME_TOTAL_LOG"),
        "BUREAU_이력(타기관대출이력)":    pref("BUREAU_"),

        # ---- B_비금융_기존 ----
        "AGE(나이)":                     pick("AGE"),
        "YEARS_EMPLOYED(근속)":          pick("YEARS_EMPLOYED"),
        "DAYS_EMPLOYED_ANOM(고용이상)":  pick("DAYS_EMPLOYED_ANOM"),
        "OCCUPATION(직업)":              pref("OCCUPATION_TYPE"),
        "ORGANIZATION(업종)":            pref("ORGANIZATION_TYPE"),
        "INCOME_TYPE(소득유형)":         pref("NAME_INCOME_TYPE"),

        # ---- C_비금융_신규 ----
        "SOCIAL_CIRCLE(사회연결망·RQ2핵심)": pick("OBS_30_CNT_SOCIAL_CIRCLE",
                                              "DEF_30_CNT_SOCIAL_CIRCLE",
                                              "SOCIAL_CIRCLE_MISSING_FLAG"),
        "FLAG_OWN_CAR(차량보유)":        pick("FLAG_OWN_CAR"),
        "NAME_EDUCATION_TYPE(교육수준)": pref("NAME_EDUCATION_TYPE_C_"),
        "NAME_HOUSING_TYPE(주거유형)":   pref("NAME_HOUSING_TYPE_C_"),
    }

    g = {k: v for k, v in g.items() if v}          # 빈 그룹 제거

    used = set(sum(g.values(), []))
    leftover = [c for c in allcols if c not in used]
    if leftover:
        print(f"⚠️ 미분류 {len(leftover)}개 → '기타'로 묶음: {leftover}")
        g["기타(미분류)"] = leftover
    return g

### 5C 프레임워크 매핑 (map_5c)

In [ ]:
def map_5c(v):
    # 사회연결망은 전통 5C 어디에도 안 맞는 게 핵심이라 강제로 끼워맞추지 않음
    if v.startswith(("OBS_30_CNT_SOCIAL_CIRCLE", "DEF_30_CNT_SOCIAL_CIRCLE")) or v == "SOCIAL_CIRCLE_MISSING_FLAG":
        return "기타(5C밖 신규축·RQ2핵심)"
    if v == "FLAG_OWN_CAR":
        return "Collateral(담보)"
    if v.startswith("NAME_EDUCATION_TYPE_C_"):
        return "Capacity(상환능력)"
    if v.startswith("NAME_HOUSING_TYPE_C_"):
        return "Collateral(담보)"
    # bureau: 과거 상환이력 = Character(상환의지) 축 보강
    if v.startswith("BUREAU_"):
        return "Character(상환의지)"

    if v.startswith(("OCCUPATION", "ORGANIZATION", "NAME_INCOME", "NAME_CONTRACT")):
        return "Conditions(여건)"
    if v == "AGE":
        return "Character(상환의지·대리)"
    if v in ["LTV_GOODS"] or "GOODS" in v or "CREDIT" in v:
        return "Collateral(담보)"
    return "Capacity(상환능력)"

### 실행 (main) — AUC 계산 → 그룹기여도 → 판정표

In [ ]:
def main():
    df, origin = load_abc(verbose=False)
    if SAMPLE_N and SAMPLE_N < len(df):
        df = df.sample(SAMPLE_N, random_state=1).reset_index(drop=True)
    y = df[TARGET].values

    s = pd.read_csv(OUT / "univariate_summary.csv")
    iv = dict(zip(s.feature, s.IV)); ext = dict(zip(s.feature, s["EXT2_상관"]))

    allcols = [c for c in df.columns if c not in (ID, TARGET)]
    groups = build_groups(allcols)

    covered = set(sum(groups.values(), []))
    print(f"표본 {len(df):,}행 / 피처 {len(allcols)}개 / 그룹 {len(groups)}개 "
          f"(분류된 피처 {len(covered)}개)")
    assert covered == set(allcols), f"미분류 컬럼 존재: {set(allcols) - covered}"

    folds = list(StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE).split(df, y))

    def auc(cols):
        if not cols:
            return 0.5
        X = df[cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        oof = np.zeros(len(y))
        for tr, va in folds:
            m = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.04, num_leaves=31,
                                   colsample_bytree=.8, subsample=.9, subsample_freq=1,
                                   verbose=-1, n_jobs=-1).fit(X.iloc[tr], y[tr])
            oof[va] = m.predict_proba(X.iloc[va])[:, 1]
        return roc_auc_score(y, oof)

    base = auc(allcols)
    print(f"전체 AUC = {base:.5f}\n그룹별 leave-one-out 측정...")

    # 개념그룹별 기여도
    rows = []
    for gname, cols in groups.items():
        a_out = auc([c for c in allcols if c not in cols])
        rows.append({"개념그룹": gname, "변수수": len(cols),
                     "군": origin.get(cols[0], "-"),
                     "그룹기여도": round(base - a_out, 5)})
    gdf = pd.DataFrame(rows).sort_values("그룹기여도", ascending=False)
    gdf.to_csv(OUT / "group_decision.csv", index=False, encoding="utf-8-sig")
    print(gdf.to_string(index=False))

    # 변수 단위 판정표
    gc_map = dict(zip(gdf.개념그룹, gdf.그룹기여도))
    out = []
    for gname, cols in groups.items():
        gcontrib = gc_map[gname]
        is_onehot = len(cols) > 1 and "중복" not in gname
        for c in cols:
            if "중복후보" in gname:                       # 원본 금액: 대표만 유지
                keep = c in ["AMT_CREDIT_LOG", "AMT_INCOME_TOTAL_LOG"]
                judg = "대표유지(규모정보)" if keep else "제거(비율변수와 중복)"
                unit = "그룹(원본금액)"
            else:
                keep = gcontrib >= 0                       # 기여도 0 이상이면 포함
                judg = ("필수" if gcontrib >= 0.002 else
                        "포함권장" if gcontrib >= 0.0003 else
                        "선택(해석용)" if gcontrib >= 0 else "제거가능")
                unit = "그룹(원핫)" if is_onehot else "개별"
            out.append({"변수": c, "개념그룹": gname.split("(")[0],
                        "군": origin.get(c, "-"), "5C_매핑": map_5c(c),
                        "IV": round(iv.get(c, np.nan), 4),
                        "EXT상관": round(ext.get(c, np.nan), 4),
                        "그룹기여도": round(gcontrib, 5), "평가단위": unit,
                        "최종판정": judg, "최종포함": "포함" if keep else "제외"})
    fdf = pd.DataFrame(out).sort_values(["최종포함", "그룹기여도"], ascending=[True, False])
    fdf.to_csv(OUT / "변수선정_최종.csv", index=False, encoding="utf-8-sig")
    print(f"\n최종: 포함 {(fdf.최종포함=='포함').sum()}개 / "
          f"제외 {(fdf.최종포함=='제외').sum()}개")
    return gdf, fdf


if __name__ == "__main__":
    gdf, fdf = main()